In [109]:
#import packages
import datetime

import pandas as pd
import numpy as np
import quandl

#to plot within notebook
import matplotlib.pyplot as plt
%matplotlib inline

#setting figure size
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 20,10

#importing required libraries
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import MinMaxScaler, Imputer
from sklearn.impute import SimpleImputer
# from keras.models import Sequential
# from keras.layers import Dense, Dropout, LSTM
from fastai.structured import add_datepart

#to get stock OHCL data
from stocker import Stocker

quandl.ApiConfig.api_key = 'VGnWgTpUun-Lz-Aeh3gt'

#### Get OHLC data for the Ticker

In [150]:
amzn_ohlc = quandl.get("WIKI/gww", start_date="2008-04-01", end_date="2017-12-31", collapse="quarterly", returns="pandas")
amzn_ohlc = amzn_ohlc.reset_index()
# amzn_ohlc['Date'] = amzn_ohlc['Date'].astype(str)  
# df1['date_time'] = pd.to_datetime(df1.date_time)
amzn_ohlc['Date'] = pd.to_datetime(amzn_ohlc.Date)
type(amzn_ohlc['Date'][0])
amzn_ohlc.head(20)

,Date,Open,High,Low,Close,Volume,Ex-Dividend,Split Ratio,Adj. Open,Adj. High,Adj. Low,Adj. Close,Adj. Volume
0,2008-06-30,81.59,82.770,81.060,81.80,667600.0,0.0,1.0,68.502228,69.492945,68.057245,68.678542,667600.0
1,2008-09-30,85.00,87.600,82.755,86.97,987600.0,0.0,1.0,71.696126,73.889184,69.802504,73.357789,987600.0
2,2008-12-31,78.09,79.580,77.400,78.84,905100.0,0.0,1.0,66.229017,67.492702,65.643820,66.865100,905100.0
3,2009-03-31,70.70,71.490,68.700,70.18,1191900.0,0.0,1.0,60.289670,60.963345,58.584163,59.846238,1191900.0
4,2009-06-30,82.83,83.110,81.120,81.88,641600.0,0.0,1.0,71.038346,71.278486,69.571781,70.223588,641600.0
5,2009-09-30,89.91,90.240,88.520,89.36,734500.0,0.0,1.0,77.508809,77.793292,76.310530,77.034670,734500.0
6,2009-12-31,97.87,98.630,96.780,96.83,526900.0,0.0,1.0,84.773912,85.432216,83.829766,83.873076,526900.0
7,2010-03-31,108.97,109.345,107.960,108.12,582300.0,0.0,1.0,94.826387,95.152714,93.947479,94.086712,582300.0
8,2010-06-30,99.74,101.440,99.220,99.45,652200.0,0.0,1.0,87.234467,88.721318,86.779665,86.980827,652200.0
9,2010-09-30,120.52,121.840,118.110,119.11,485000.0,0.0,1.0,105.907955,107.067916,103.790147,104.668905,485000.0


#### Read the 10-K and 10-Q sentitment scores for the Ticker

In [111]:
#read sentiments for Amazon
columns = ['Cik','Coname','Date','Form','Secname','Neg_Score','Neu_Score','Pos_Score']

#read 10Q sentiments
amzn_10q_sentiments = pd.read_csv('/Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/sentiment-scores/10-Q/0000277135.csv',names=columns)
amzn_10q_sentiments.sort_values(by=['Date'], inplace=True)

#read 10K sentiments
amzn_10k_sentiments = pd.read_csv('/Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/sentiment-scores/10-K/0000277135.csv',names=columns)
amzn_10k_sentiments.sort_values(by=['Date'], inplace=True)


#### Merge the 10-K and 10-Q dataframes together and sort them on date basis

In [112]:
amzn_all_sentiments = pd.concat([amzn_10q_sentiments,amzn_10k_sentiments], ignore_index=True)
amzn_all_sentiments.sort_values(by=['Date'], inplace=True)
# amzn_all_sentiments = amzn_all_sentiments[amzn_all_sentiments.FDATE > '2008-04-01' and amzn_all_sentiments.FDATE < '2017-12-31']
amzn_all_sentiments['Date'] = pd.to_datetime(amzn_all_sentiments.Date)
type(amzn_all_sentiments['Date'][0])
amzn_all_sentiments.head(10)

,Cik,Coname,Date,Form,Secname,Neg_Score,Neu_Score,Pos_Score
33,277135,gww,2008-02-27,10-K,0000277135_2008-02-27.txt,0.039250,0.832313,0.128500
0,277135,gww,2008-05-08,10-Q,0000277135_2008-05-08.txt,0.028667,0.856333,0.114667
1,277135,gww,2008-07-31,10-Q,0000277135_2008-07-31.txt,0.032000,0.869200,0.098800
2,277135,gww,2008-10-30,10-Q,0000277135_2008-10-30.txt,0.034400,0.871600,0.094000
34,277135,gww,2009-02-27,10-K,0000277135_2009-02-27.txt,0.037000,0.845313,0.117813
3,277135,gww,2009-05-04,10-Q,0000277135_2009-05-04.txt,0.035750,0.859000,0.104750
4,277135,gww,2009-07-31,10-Q,0000277135_2009-07-31.txt,0.043200,0.857800,0.099000
5,277135,gww,2009-10-29,10-Q,0000277135_2009-10-29.txt,0.036800,0.860800,0.102400
35,277135,gww,2010-02-25,10-K,0000277135_2010-02-25.txt,0.036688,0.847375,0.116000
6,277135,gww,2010-04-29,10-Q,0000277135_2010-04-29.txt,0.023667,0.860000,0.116000


#### Mapping quarterly OHLC data for chosen tickers with the 10K and 10Q sec filing's sentiment scores using a forward window and visualizing the insights to analyze the effect of sec sentiments on the stock prices.

In [113]:
amzn_sentimnts_ohlc = pd.merge_asof(amzn_all_sentiments, amzn_ohlc, on='Date', direction='forward')
amzn_sentimnts_ohlc = amzn_sentimnts_ohlc.dropna()
amzn_sentimnts_ohlc.head(10)

,Cik,Coname,Date,Form,Secname,Neg_Score,Neu_Score,Pos_Score,Open,High,Low,Close,Volume,Ex-Dividend,Split Ratio,Adj. Open,Adj. High,Adj. Low,Adj. Close,Adj. Volume
0,277135,gww,2008-02-27,10-K,0000277135_2008-02-27.txt,0.039250,0.832313,0.128500,81.59,82.770,81.060,81.80,667600.0,0.0,1.0,68.502228,69.492945,68.057245,68.678542,667600.0
1,277135,gww,2008-05-08,10-Q,0000277135_2008-05-08.txt,0.028667,0.856333,0.114667,81.59,82.770,81.060,81.80,667600.0,0.0,1.0,68.502228,69.492945,68.057245,68.678542,667600.0
2,277135,gww,2008-07-31,10-Q,0000277135_2008-07-31.txt,0.032000,0.869200,0.098800,85.00,87.600,82.755,86.97,987600.0,0.0,1.0,71.696126,73.889184,69.802504,73.357789,987600.0
3,277135,gww,2008-10-30,10-Q,0000277135_2008-10-30.txt,0.034400,0.871600,0.094000,78.09,79.580,77.400,78.84,905100.0,0.0,1.0,66.229017,67.492702,65.643820,66.865100,905100.0
4,277135,gww,2009-02-27,10-K,0000277135_2009-02-27.txt,0.037000,0.845313,0.117813,70.70,71.490,68.700,70.18,1191900.0,0.0,1.0,60.289670,60.963345,58.584163,59.846238,1191900.0
5,277135,gww,2009-05-04,10-Q,0000277135_2009-05-04.txt,0.035750,0.859000,0.104750,82.83,83.110,81.120,81.88,641600.0,0.0,1.0,71.038346,71.278486,69.571781,70.223588,641600.0
6,277135,gww,2009-07-31,10-Q,0000277135_2009-07-31.txt,0.043200,0.857800,0.099000,89.91,90.240,88.520,89.36,734500.0,0.0,1.0,77.508809,77.793292,76.310530,77.034670,734500.0
7,277135,gww,2009-10-29,10-Q,0000277135_2009-10-29.txt,0.036800,0.860800,0.102400,97.87,98.630,96.780,96.83,526900.0,0.0,1.0,84.773912,85.432216,83.829766,83.873076,526900.0
8,277135,gww,2010-02-25,10-K,0000277135_2010-02-25.txt,0.036688,0.847375,0.116000,108.97,109.345,107.960,108.12,582300.0,0.0,1.0,94.826387,95.152714,93.947479,94.086712,582300.0
9,277135,gww,2010-04-29,10-Q,0000277135_2010-04-29.txt,0.023667,0.860000,0.116000,99.74,101.440,99.220,99.45,652200.0,0.0,1.0,87.234467,88.721318,86.779665,86.980827,652200.0


In [114]:
indexed_amzn_sentimnts_ohlc = amzn_sentimnts_ohlc.set_index(amzn_sentimnts_ohlc.Date)
indexed_amzn_sentimnts_ohlc.head(5)
type(indexed_amzn_sentimnts_ohlc['Date'][0])

pandas._libs.tslibs.timestamps.Timestamp

In [149]:
amzn_sentimnts_ohlc.head(20)

,Cik,Coname,Date,Form,Secname,Neg_Score,Neu_Score,Pos_Score,Open,High,Low,Close,Volume,Ex-Dividend,Split Ratio,Adj. Open,Adj. High,Adj. Low,Adj. Close,Adj. Volume
0,277135,gww,2008-02-27,10-K,0000277135_2008-02-27.txt,0.039250,0.832313,0.128500,81.59,82.770,81.060,81.80,667600.0,0.0,1.0,68.502228,69.492945,68.057245,68.678542,667600.0
1,277135,gww,2008-05-08,10-Q,0000277135_2008-05-08.txt,0.028667,0.856333,0.114667,81.59,82.770,81.060,81.80,667600.0,0.0,1.0,68.502228,69.492945,68.057245,68.678542,667600.0
2,277135,gww,2008-07-31,10-Q,0000277135_2008-07-31.txt,0.032000,0.869200,0.098800,85.00,87.600,82.755,86.97,987600.0,0.0,1.0,71.696126,73.889184,69.802504,73.357789,987600.0
3,277135,gww,2008-10-30,10-Q,0000277135_2008-10-30.txt,0.034400,0.871600,0.094000,78.09,79.580,77.400,78.84,905100.0,0.0,1.0,66.229017,67.492702,65.643820,66.865100,905100.0
4,277135,gww,2009-02-27,10-K,0000277135_2009-02-27.txt,0.037000,0.845313,0.117813,70.70,71.490,68.700,70.18,1191900.0,0.0,1.0,60.289670,60.963345,58.584163,59.846238,1191900.0
5,277135,gww,2009-05-04,10-Q,0000277135_2009-05-04.txt,0.035750,0.859000,0.104750,82.83,83.110,81.120,81.88,641600.0,0.0,1.0,71.038346,71.278486,69.571781,70.223588,641600.0
6,277135,gww,2009-07-31,10-Q,0000277135_2009-07-31.txt,0.043200,0.857800,0.099000,89.91,90.240,88.520,89.36,734500.0,0.0,1.0,77.508809,77.793292,76.310530,77.034670,734500.0
7,277135,gww,2009-10-29,10-Q,0000277135_2009-10-29.txt,0.036800,0.860800,0.102400,97.87,98.630,96.780,96.83,526900.0,0.0,1.0,84.773912,85.432216,83.829766,83.873076,526900.0
8,277135,gww,2010-02-25,10-K,0000277135_2010-02-25.txt,0.036688,0.847375,0.116000,108.97,109.345,107.960,108.12,582300.0,0.0,1.0,94.826387,95.152714,93.947479,94.086712,582300.0
9,277135,gww,2010-04-29,10-Q,0000277135_2010-04-29.txt,0.023667,0.860000,0.116000,99.74,101.440,99.220,99.45,652200.0,0.0,1.0,87.234467,88.721318,86.779665,86.980827,652200.0


In [116]:
test = pd.DataFrame(indexed_amzn_sentimnts_ohlc, columns=['Neg_Score', 'Pos_Score']) #.pct_change()
test_one =  pd.DataFrame(indexed_amzn_sentimnts_ohlc, columns=['Close'])
test = test.reset_index()
test_one= test_one.reset_index()

#### Analyze the trend between stock price and positive sentiments

In [146]:
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, iplot
init_notebook_mode(connected=True)

# Create traces
trace0 = go.Scatter(
    x = test['Date'],
    y = np.log(test['Pos_Score']),
    mode = 'lines+markers',
    name='Positive Sentiments'
)

trace1 = go.Scatter(
    x =  test_one['Date'],
    y =  test_one['Close'],
    mode = 'lines+markers',
    name = 'Stock`s Closing Price',
    yaxis='y2'
)

layout = go.Layout(
    xaxis=dict(tickangle=-45),
    yaxis=dict(
        autorange=True,
        title = 'Positive Sentiments'
   ),    
    
    legend=dict(
        x=0,
        y=1.2,
        traceorder='normal',
        font=dict(
            family='sans-serif',
            size=12,
            color='#000'
        ),
        bgcolor='#E2E2E2',
        bordercolor='#FFFFFF',
        borderwidth=2
    ),
    yaxis2=dict(
        title='Stock`s Closing Price',
        overlaying='y',
        side='right'
    ),
)

data = [trace0, trace1]
fig = go.Figure(data=data,layout=layout)
iplot(fig)

#### Analyze the trend between stock price and negative sentiments

In [147]:
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, iplot
init_notebook_mode(connected=True)

# Create traces
trace0 = go.Scatter(
    x = test['Date'],
    y = np.log(test['Neg_Score']),
    mode = 'lines+markers',
    name='Negative Sentiments'
)

trace1 = go.Scatter(
    x =  test_one['Date'],
    y =  test_one['Close'],
    mode = 'lines+markers',
    name = 'Stock`s Closing Price',
    yaxis='y2'
)

layout = go.Layout(
    xaxis=dict(tickangle=-45),
    yaxis=dict(
        autorange=True,
        title = 'Negative Sentiments'
   ),    
    legend=dict(
        x=0,
        y=1.2,
        traceorder='normal',
        font=dict(
            family='sans-serif',
            size=12,
            color='#000'
        ),
        bgcolor='#E2E2E2',
        bordercolor='#FFFFFF',
        borderwidth=2
    ),
    yaxis2=dict(
        title='Stock`s Closing Price',
        overlaying='y',
        side='right'
    ),
)
data = [trace0, trace1]
fig = go.Figure(data=data,layout=layout)
iplot(fig)